# Reproducible scRNA-seq workflow: restartable checkpoints

Run all cells top to bottom, or restart from any stage. Each stage reads **only the previous stage's saved checkpoint**, writes its own checkpoint plus a PASS/WARN/FAIL report, and records what it used in `manifest.json`. State lives in files, not in notebook memory, so rerunning a stage never uses stale variables.

**How to go back and restart**
- *Something looks wrong in stage N:* change the stage's parameters (setup cell), rerun the notebook. Stages before N load from their checkpoints; stage N and everything after it recompute.
- *Force one stage to recompute:* `run_stage("qc", stages.qc, stages.write_qc, force=True)`.
- *Forget a stage and everything after it:* `wf.invalidate("qc")`.
- *A stage reported FAIL:* the run stops there (checkpoint kept for inspection). Fix the cause, or continue on purpose with `wf.accept("qc", "reason")`; the reason is stored in `manifest.json`.
- *Which stages are current or stale?* The status table at the end of every stage section.

Stages in this notebook: 0 fetch, 1 assemble (integrity checks), 2 QC and filtering, 3 normalization and feature selection, 4 dimensionality reduction, 5 clustering, 6 cell type annotation, 7 condition comparison, 8 biological interpretation and validation. Later stages (group comparisons) are placeholders at the end.

In [ ]:
import os, sys, json
from pathlib import Path
HERE = Path.cwd()
sys.path.insert(0, str(HERE / "scrna_workflow"))
import stages
from ckpt import Workflow, StageFailed
try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = str

# ---- choose the dataset: "baron" (small, UMI counts), "sade_feldman" (large, Smart-seq2 TPM) or "kang" (control vs interferon-beta, 10x counts) or "paul15" (mouse bone marrow, MARS-seq)
DATASET = os.environ.get("NB_DATASET", "baron")
RUN = HERE / "runs" / DATASET
GEO = "https://ftp.ncbi.nlm.nih.gov/geo"

IFN_GENES = ['ISG15', 'IFIT1', 'IFIT2', 'IFIT3', 'MX1', 'MX2', 'OAS1', 'OAS2', 'OAS3', 'OASL', 'IFI6', 'IFI27', 'IFI44', 'IFI44L', 'IFITM1', 'IFITM3', 'ISG20', 'IRF7', 'RSAD2', 'STAT1', 'XAF1', 'HERC5', 'USP18', 'EPSTI1', 'LY6E', 'SAMD9L', 'CMPK2', 'TRIM22']   # interferon-response signature supplied by the analyst (test set only)

PARAMS = {
    "baron": {
        "global": {"raw_dir": str(RUN / "raw"), "run_dir": str(RUN), "species": "human"},
        "fetch": {"files": [{"name": "matrix", "filename": "GSM2230757_human1_umifm_counts.csv.gz",
                             "url": f"{GEO}/samples/GSM2230nnn/GSM2230757/suppl/GSM2230757_human1_umifm_counts.csv.gz"}]},
        "assemble": {"adapter": "baron_csv", "declared_scale": "counts",
                     # expected numbers: replace with the paper's numbers when you have them (these were read from the file)
                     "expected": {"shape": [1937, 20125],
                                  "obs_counts": {"assigned_cluster": {"beta": 872, "alpha": 236, "delta": 214, "endothelial": 130,
                                                 "ductal": 120, "acinar": 110, "quiescent_stellate": 92, "gamma": 70,
                                                 "activated_stellate": 51, "macrophage": 14, "epsilon": 13, "mast": 8,
                                                 "schwann": 5, "t_cell": 2}}}},
        "qc": {"min_genes": 200, "max_genes": None, "max_pct_mt": 20, "min_cells": 3, "sample_col": None},
        "features": {"n_top_genes": 2000, "batch_key": None, "sample_col": None},
        "dimred": {"batch_key": None, "variance_threshold": 0.9},   # batch_key: a batch column turns on the batch checks and, if needed, Harmony correction
        "cluster": {"batch_key": None, "resolution": None},        # resolution None = chosen from the data; use the same batch_key as dimred
        "annotate": {"panel": str(RUN / "marker_panel.csv"),        # editable CSV: cell_type, positive, negative, source (genes separated by ;)
                     "reference": None,   # optional labelled .h5ad (None to skip); a different donor here
                     "reference_label_key": "cell_type",
                     "label_map": {"quiescent_stellate": "pancreatic stellate cell", "activated_stellate": "pancreatic stellate cell", "t_cell": "T cell",
                                   "schwann": "Schwann cell", "gamma": "pancreatic PP cell", "epsilon": "pancreatic epsilon cell",
                                   "delta": "pancreatic delta cell"}},
        "compare": {"condition": None},      # one donor, no groups to compare
        "interpret": {"gene_sets": str(RUN / "gene_sets" / "kegg_hsa.gmt"), "gene_sets_kegg": "hsa",   # KEGG pathways fetched once through rest.kegg.jp; or point to any GMT
                      "expectations": str(RUN / "expectations.csv"), "signature": None, "perturb": True, "max_cells": 3000},
    },
    "sade_feldman": {
        "global": {"raw_dir": str(RUN / "raw"), "run_dir": str(RUN), "species": "human"},
        "fetch": {"files": [
            {"name": "matrix", "filename": "GSE120575_TPM.txt.gz",
             "url": f"{GEO}/series/GSE120nnn/GSE120575/suppl/GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz"},
            {"name": "meta", "filename": "GSE120575_patient_ID.txt.gz",
             "url": f"{GEO}/series/GSE120nnn/GSE120575/suppl/GSE120575_patient_ID_single_cells.txt.gz"}]},
        "assemble": {"adapter": "sade_feldman", "declared_scale": "log2tpm", "chunk_rows": 1000,
                     "expected": {"shape": [16291, 55737],
                                  "obs_counts": {"timepoint": {"Post": 10363, "Pre": 5928},
                                                 "response": {"Non-responder": 10727, "Responder": 5564}},
                                  "n_unique": {"patient": 32, "patient_timepoint": 48}}},
        "qc": {"min_genes": 200, "max_genes": None, "max_pct_mt": 40, "min_cells": 3, "sample_col": "patient_timepoint"},
        # batch_key: set to a donor/batch column (e.g. "patient") when there are several batches
        "features": {"n_top_genes": 2000, "batch_key": None, "sample_col": "patient_timepoint"},
        "dimred": {"batch_key": None, "variance_threshold": 0.9},   # e.g. batch_key "patient"; beware batches that are confounded with biology
        "cluster": {"batch_key": None, "resolution": None},        # same batch_key as dimred; resolution None = chosen from the data
        "annotate": {"panel": str(RUN / "marker_panel.csv"), "reference": None, "reference_label_key": "cell_type", "label_map": {}},   # create marker_panel.csv first
        "compare": {"condition": None},      # to compare responders with non-responders: set condition, sample (patient) and groups after checking the obs columns
        "interpret": {"gene_sets": str(RUN / "gene_sets" / "kegg_hsa.gmt"), "gene_sets_kegg": "hsa", "expectations": None, "signature": None, "perturb": True, "max_cells": 3000},
    },
    "kang": {   # GSE96583 batch 2: PBMC from 8 donors, control vs interferon-beta (10x counts); test set for the condition comparison
        "global": {"raw_dir": str(RUN / "raw"), "run_dir": str(RUN), "species": "human"},
        "fetch": {"files": [
            {"name": "mtx_ctrl", "filename": "GSM2560248_2.1.mtx.gz", "url": f"{GEO}/samples/GSM2560nnn/GSM2560248/suppl/GSM2560248_2.1.mtx.gz"},
            {"name": "bc_ctrl", "filename": "GSM2560248_barcodes.tsv.gz", "url": f"{GEO}/samples/GSM2560nnn/GSM2560248/suppl/GSM2560248_barcodes.tsv.gz"},
            {"name": "mtx_stim", "filename": "GSM2560249_2.2.mtx.gz", "url": f"{GEO}/samples/GSM2560nnn/GSM2560249/suppl/GSM2560249_2.2.mtx.gz"},
            {"name": "bc_stim", "filename": "GSM2560249_barcodes.tsv.gz", "url": f"{GEO}/samples/GSM2560nnn/GSM2560249/suppl/GSM2560249_barcodes.tsv.gz"},
            {"name": "genes", "filename": "GSE96583_batch2.genes.tsv.gz", "url": f"{GEO}/series/GSE96nnn/GSE96583/suppl/GSE96583_batch2.genes.tsv.gz"},
            {"name": "meta", "filename": "GSE96583_batch2.total.tsne.df.tsv.gz", "url": f"{GEO}/series/GSE96nnn/GSE96583/suppl/GSE96583_batch2.total.tsne.df.tsv.gz"}]},
        "assemble": {"adapter": "kang_mtx", "declared_scale": "counts", "max_cells": 10000, "seed": 0,     # max_cells: random subsample for small machines; None keeps all singlets
                     "expected": {"n_unique": {"sample": 8, "condition": 2}}},
        "qc": {"min_genes": 200, "max_genes": None, "max_pct_mt": 20, "min_cells": 3, "sample_col": "sample"},
        "features": {"n_top_genes": 2000, "batch_key": None, "sample_col": "sample"},
        "dimred": {"batch_key": None, "variance_threshold": 0.9},   # library is confounded with condition here, so it is not used as a batch
        "cluster": {"batch_key": None, "resolution": None},
        "annotate": {"panel": str(RUN / "marker_panel.csv"), "reference": None, "reference_label_key": "cell_type", "label_map": {}},
        "compare": {"condition": "condition", "sample": "sample", "batch": "library", "groups": ["ctrl", "stim"],     # control first; sample = donor, library = batch
                    "gene_sets": str(RUN / "gene_sets" / "kegg_hsa.gmt"), "gene_sets_kegg": "hsa", "min_units": 3, "min_cells": 10, "fdr": 0.05, "min_lfc": 0.5,
                    "state": True, "n_swaps": 20, "question": "What changes in blood cells when interferon-beta (stim) is compared with untreated cells (ctrl) from the same donors?"},
        "interpret": {"gene_sets": str(RUN / "gene_sets" / "kegg_hsa.gmt"), "gene_sets_kegg": "hsa", "expectations": str(RUN / "expectations.csv"),
                      "signature": {"name": "Interferon signature", "genes": IFN_GENES, "label_col": "condition", "positive": "stim", "sample_col": "sample", "by_type": True},
                      "perturb": True, "max_cells": 3000},
    },
    "paul15": {   # GSE72857 (Paul 2015): mouse bone marrow, MARS-seq UMI counts; wild-type experiments only (sorted subsets and unsorted myeloid)
        "global": {"raw_dir": str(RUN / "raw"), "run_dir": str(RUN), "species": "mouse"},
        "fetch": {"files": [
            {"name": "matrix", "filename": "GSE72857_umitab.txt.gz", "url": f"{GEO}/series/GSE72nnn/GSE72857/suppl/GSE72857_umitab.txt.gz"},
            {"name": "meta", "filename": "GSE72857_experimental_design.txt.gz", "url": f"{GEO}/series/GSE72nnn/GSE72857/suppl/GSE72857_experimental_design.txt.gz"}]},
        "assemble": {"adapter": "paul15_umitab", "declared_scale": "counts", "chunk_rows": 2000,
                     "keep_batches": ["Unsorted myeloid", "CMP CD41", "CMP Flt3+ Csf1r+", "CMP Irf8-GFP+ MHCII+"],   # Cebpa/Cebpe knockouts and controls left out: one mouse per genotype, so no replicate for a genotype comparison
                     # expected numbers were read from the files (10,368 wells; 108 flagged empty; wild-type experiments 5,700 single-cell wells), not taken from the paper
                     "expected": {"shape": [5700, 27297], "n_unique": {"sort_scheme": 4, "mouse": 4}}},
        "qc": {"min_genes": 200, "max_genes": None, "max_pct_mt": 20, "min_cells": 3, "sample_col": "mouse"},
        "features": {"n_top_genes": 2000, "batch_key": None, "sample_col": "mouse"},
        "dimred": {"batch_key": None, "variance_threshold": 0.9},   # mouse is confounded with the sorting scheme (each sorted subset came from its own mouse), so it is not used as a batch
        "cluster": {"batch_key": None, "resolution": None},
        "annotate": {"panel": str(RUN / "marker_panel.csv"), "reference": None, "reference_label_key": "cell_type", "label_map": {}},
        "compare": {"condition": None},      # no condition with replicate samples among the wild-type experiments
        "interpret": {"gene_sets": str(RUN / "gene_sets" / "kegg_mmu.gmt"), "gene_sets_kegg": "mmu", "expectations": str(RUN / "expectations.csv"),
                      "signature": None, "perturb": True, "max_cells": 3000},
    },
}[DATASET]

# the panel and reference files are inputs: record their contents in the stage parameters so editing a file reruns the stage
from ckpt import sha256_file
for _stage, _keys in (("annotate", ("panel", "reference")), ("compare", ("gene_sets",)), ("interpret", ("gene_sets", "expectations"))):
    for _k in _keys:
        _p = PARAMS[_stage].get(_k)
        PARAMS[_stage][_k + "_sha256"] = sha256_file(_p) if _p and Path(_p).exists() else None

STAGES = ["fetch", "assemble", "qc", "features", "dimred", "cluster", "annotate", "compare", "interpret"]
wf = Workflow(RUN, PARAMS, STAGES, kinds={"fetch": "json"})


# Deliberate decisions to continue past a FAIL, written down before the run: {stage: reason}. The reason is stored in manifest.json and RUN_SUMMARY.md.
ACCEPT = {
    "baron": {"qc": "GSE84133 has no mitochondrial genes, so the mitochondrial filter does not apply"},
}.get(DATASET, {})


def run_stage(name, fn, writer, force=False):
    """Run (or load) a stage and always show its report, even when it failed. A FAIL stops the run unless it is listed in ACCEPT."""
    try:
        try:
            return wf.run(name, fn, writer, force=force)
        except StageFailed:
            if name in ACCEPT and wf.m.get(name, {}).get("status") == "FAIL":
                wf.accept(name, ACCEPT[name])
                print(f"[{name}] FAIL accepted on purpose: {ACCEPT[name]}")
                wf.summary()
                return wf.load(name)
            raise
    finally:
        rp = RUN / "reports" / f"{name}.md"
        if rp.exists():
            display(Markdown(rp.read_text()))


def show_figure(path, legend=""):
    """Show a figure with its one-line caption (what is plotted) and, if given, its plain-language legend (how to read it)."""
    import json as _json
    path = Path(path)
    capf = path.parent / "figure_captions.json"
    cap = _json.load(open(capf, encoding="utf-8")).get(path.name, "") if capf.exists() else ""
    text = f"![{path.stem}]({path.relative_to(HERE).as_posix()})\n\n**{path.stem.replace('_', ' ')}.** {cap}"
    if legend:
        text += f"\n\n*{legend}*"
    display(Markdown(text))


print("dataset:", DATASET, "| run folder:", RUN)
wf.status()

## Stage 0: fetch
Downloads the raw files (skips files already present) and checks each one: size, SHA-256 (recorded for `DATA_SOURCES.md`), and a full decompression pass that catches truncated downloads.
**If this fails:** delete the file in `runs/<dataset>/raw`, rerun. Never edit raw files by hand.

In [ ]:
files = run_stage("fetch", stages.fetch, stages.write_fetch)
wf.status()

## Stage 1: assemble and data integrity (checkpoint 1)
Parses the raw files into a sparse cells x genes matrix (large text matrices are read in chunks, never as a dense table), then checks: cell IDs unique and matching the metadata, **column order against the matrix file's own label row** (catches a shifted header), no NaN/inf, what the values are (counts, TPM, log) against what the source says, and shape and label counts against the published numbers.
**If this fails:** the report says which check and why. Do not continue until you understand it.

In [ ]:
adata = run_stage("assemble", stages.assemble, stages.write_assemble)
print(adata)
wf.status()

## Stage 2: QC and filtering (checkpoint 2)
Validates the mitochondrial gene set (`MT-` genes only; nuclear MTRNR2L pseudogenes excluded), computes per-cell metrics in row blocks, applies the filters from `PARAMS["qc"]`, and reports cells removed per filter, in total and per sample, plus genes removed. The threshold plot is saved to `runs/<dataset>/figures/qc_before_filtering.png`.
**A dataset with no mitochondrial genes fails here on purpose:** the mito filter cannot work. If that is acceptable, record it with `wf.accept("qc", "reason")` and the run continues.

In [ ]:
adata_qc = run_stage("qc", stages.qc, stages.write_qc)
print(adata_qc)
fig = RUN / "figures" / "qc_before_filtering.png"
if fig.exists():
    show_figure(fig)
wf.status()

## Stage 3: normalization and feature selection (checkpoint 3)
Can expression profiles be compared between cells, and which genes carry informative variation? The normalization is chosen from the scale detected at stage 1 (counts: scale to 10,000 per cell and log1p; TPM: log1p; log2 TPM and log-normalized values are left as they are, with the log base recorded). Then 2,000 highly variable genes are selected (`seurat_v3` on raw counts, `seurat` on log values; per batch if `batch_key` is set). The report checks per-cell totals, depth spread, dominant-gene cells, technical genes among the selection, low-detection genes and the stability of the selection on two halves of the cells. The mean-variance plot is saved to `runs/<dataset>/figures/mean_variance.png`.
**Warnings here are for review, not automatic faults:** datasets with rare cell types legitimately select genes detected in few cells.

In [ ]:
adata_feat = run_stage("features", stages.features, stages.write_features)
print(adata_feat)
fig = RUN / "figures" / "mean_variance.png"
if fig.exists():
    show_figure(fig)
wf.status()

## Stage 4: dimensionality reduction (checkpoint 4)
The selected genes are scaled and summarized by principal components (PCs). **The number of PCs is chosen from the data:** the smallest number that captures 90% of the real structure, meaning the variation that rises above what shuffled gene values would give (`variance_threshold` sets the share). The report also lists the shuffled-data count and the elbow as cross-checks, tests whether PCs track depth, genes detected or mitochondrial share, and, if `batch_key` is set, measures batch structure and applies Harmony correction when batch explains more than 20% of the retained variation. A neighbor graph and a UMAP follow.
Four figures are written to `runs/<dataset>/figures/dimred/`, each with a plain-language legend for readers who do not know these methods. **UMAP is for viewing only:** distances between groups and group sizes are not quantitative.

In [ ]:
adata_dr = run_stage("dimred", stages.dimred, stages.write_dimred)
print(adata_dr)
fdir = RUN / "figures" / "dimred"
legends = {}
if (fdir / "figure_legends.md").exists():
    for line in (fdir / "figure_legends.md").read_text(encoding="utf-8").splitlines():
        if line.startswith("**") and "): " in line:
            legends[line.split("(")[1].split(".png")[0]] = line.split("): ", 1)[1]
for key in ["pca_variance", "pca_covariates", "pca_scatter", "umap"]:
    f = fdir / f"{key}.png"
    if f.exists():
        show_figure(f, legends.get(key, ''))
wf.status()

## Stage 5: clustering (checkpoint 5)
Cells are grouped on the same PCs stage 4 chose (the Harmony PCs if batch was corrected). **The resolution is chosen from the data:** Leiden is run at eight resolutions, and the finest one is used whose clusters stay the same when 20% of cells are left out (stability of at least 0.8), that has no cluster under 10 cells (or 0.5% of cells) and no cluster holding more than half the cells, and whose neighboring clusters differ by at least 10 genes. If none qualifies, the most stable resolution is used and the report says WARN. The report also checks each cluster's recovery under resampling, finds marker genes for every cluster, and, if `batch_key` is set, flags clusters made almost entirely of one batch.
Four figures with plain-language legends are written to `runs/<dataset>/figures/cluster/`, with `cluster_markers.csv` and `resolution_scan.csv`. Clusters are numbered groups, not cell types; naming them is a later stage. **A tiny cluster WARN can be a real rare cell type:** look at its markers before merging it.

In [ ]:
adata_cl = run_stage("cluster", stages.cluster, stages.write_cluster)
print(adata_cl)
print(adata_cl.obs["cluster"].value_counts().sort_index().to_string())
fdir = RUN / "figures" / "cluster"
legends = {}
if (fdir / "figure_legends.md").exists():
    for line in (fdir / "figure_legends.md").read_text(encoding="utf-8").splitlines():
        if line.startswith("**") and "): " in line:
            legends[line.split("(")[1].split(".png")[0]] = line.split("): ", 1)[1]
for key in ["cluster_resolution", "cluster_umap", "cluster_sizes", "cluster_markers"]:
    f = fdir / f"{key}.png"
    if f.exists():
        show_figure(f, legends.get(key, ''))
wf.status()

## Stage 6: cell type annotation (checkpoint 6)
Each cluster is scored against a **marker panel you provide** (`runs/<dataset>/marker_panel.csv`, columns `cell_type`, `positive`, `negative`, `source`, genes separated by `;`). The panel can be drafted from the CellGuide connector (see the `scrna-cell-type-annotation` skill) but **read and edit it before trusting it:** some cell types return few or no usable markers. Negative markers are used only if you write them in.
An optional labelled reference dataset (a different donor or study, `.h5ad` with labels in `obs`) gives an independent second opinion. **Every cluster gets a confidence tier:** confident (one type clearly ahead, positives present, negatives off, reference agrees), probable, ambiguous (two types close, markers conflict, or the reference disagrees) or unassigned (nothing fits). Ambiguous and unassigned clusters are kept as such with the reason, because mixed, transitional or unmatched clusters are expected; a name means "matches this panel", not "proven to be this cell type".
Three figures with plain-language legends are written to `runs/<dataset>/figures/annotate/`, with `annotation_table.csv` (scores, reasons, reference labels) and the panel used. Editing the panel or reference file reruns this stage only.

In [ ]:
adata_ann = run_stage("annotate", stages.annotate, stages.write_annotate)
print(adata_ann)
print(adata_ann.uns["annotation"][["cluster", "n_cells", "label", "tier", "candidates", "reason"]].to_string(index=False))
fdir = RUN / "figures" / "annotate"
legends = {}
if (fdir / "figure_legends.md").exists():
    for line in (fdir / "figure_legends.md").read_text(encoding="utf-8").splitlines():
        if line.startswith("**") and "): " in line:
            legends[line.split("(")[1].split(".png")[0]] = line.split("): ", 1)[1]
for key in ["annotation_scores", "annotation_umap", "annotation_markers"]:
    f = fdir / f"{key}.png"
    if f.exists():
        show_figure(f, legends.get(key, ''))
wf.status()

## Stage 7: control vs treatment comparison (checkpoint 7)
This stage asks **what changes between two conditions** and starts by asking whether the data can answer that at all. **(1) Design check:** are there at least three independent samples (donors, patients) per condition, are the same samples measured in both conditions (paired), is the condition confounded with a batch or library, do the clusters or the cell-type names simply follow the condition? **(2) Abundance:** which cell types take a larger or smaller share of each sample (two models that must agree). **(3) Differential expression:** genes compared within each cell type on per-sample totals with a negative-binomial model (pseudobulk), with a label-swap check and a check that genes move the same way in individual samples. **(4) State change:** can cells of one type be told apart by condition in a donor the classifier never saw. **(5) Pathways:** pre-ranked enrichment of each type's ranked genes.
A dataset without a condition (`"condition": None`) records that nothing was requested and passes the object on. Transcription-factor activity, cell-cell communication and spatial association are listed under Not run: they need resources that are not installed (OmniPath, a ligand-receptor table, tissue coordinates).
**Read the design checks before the gene lists:** if every library holds one condition, a treatment effect and a library effect cannot be told apart.

In [ ]:
adata_cmp = run_stage("compare", stages.compare, stages.write_compare)
print(adata_cmp)
fdir = RUN / "figures" / "compare"
if PARAMS["compare"].get("condition"):
    legends = {}
    if (fdir / "figure_legends.md").exists():
        for line in (fdir / "figure_legends.md").read_text(encoding="utf-8").splitlines():
            if line.startswith("**") and "): " in line:
                legends[line.split("(")[1].split(".png")[0]] = line.split("): ", 1)[1]
    for key in ["cmp_abundance", "cmp_de", "cmp_state", "cmp_pathways"]:
        f = fdir / f"{key}.png"
        if f.exists():
            show_figure(f, legends.get(key, ""))
    for k, v in json.loads((RUN / "reports" / "compare.json").read_text(encoding="utf-8")).get("answers", {}).items():
        print(f"{k}: {v}")
wf.status()

## Stage 8: biological interpretation and validation (checkpoint 8)
The final stage asks what the named cell types mean, how much to trust that, and what the data cannot say. It runs: **(1) pathway analysis**, testing each cell type's marker genes for over-representation of gene sets against the genes actually detected here (KEGG pathways are fetched once through `rest.kegg.jp`, or point `gene_sets` at any GMT file); **(2) gene programs**, found by the data itself with non-negative matrix factorisation and kept only if they are reproducible across random restarts; **(3) validation against expectations**, a small file of known biology and negative controls that the data must reproduce, plus (when a signature is given) a leave-one-sample-out test that the signature separates two groups in samples it never saw; **(4) robustness and sensitivity**, re-running the variable genes, PCA, clustering and naming under changed seeds, cell subsets, gene counts, PC counts and resolution, and labelling each cell type robust, sensitive or unstable. The report ends with a findings table (finding, evidence, robustness, caveats) and carries every warning and unrun check from earlier stages into the conclusions.
**Read the robustness labels before the pathway names:** a pathway name describes genes the markers share, not an active pathway, and a cell type labelled sensitive or unstable depends on a choice that could reasonably have been made differently.

In [ ]:
adata_int = run_stage("interpret", stages.interpret, stages.write_interpret)
print(adata_int)
fdir = RUN / "figures" / "interpret"
legends = {}
if (fdir / "figure_legends.md").exists():
    for line in (fdir / "figure_legends.md").read_text(encoding="utf-8").splitlines():
        if line.startswith("**") and "): " in line:
            legends[line.split("(")[1].split(".png")[0]] = line.split("): ", 1)[1]
for key in ["int_pathways", "int_programs", "int_robustness", "int_validation"]:
    f = fdir / f"{key}.png"
    if f.exists():
        show_figure(f, legends.get(key, ""))
if "interpretation_findings" in adata_int.uns:
    print(adata_int.uns["interpretation_findings"][["finding", "robustness"]].to_string(index=False, max_colwidth=90))
wf.status()

## Later stages: not built yet
7. Group comparisons and signatures.

It will follow the same pattern: read the previous checkpoint, write its own, report PASS/WARN/FAIL with reasons, list what ran and what did not.

## Run summary
One page for the whole run, rebuilt after every stage: `runs/<dataset>/RUN_SUMMARY.md`. It lists, per stage, the verdict and the reason, every check that ran with what it tests and its result, the checks that did not run, the parameters used, software versions, timing and memory, the input-file hashes, all figures with their legends, and how to go back and resume. Share this file with the figures folder to show the whole analysis.

In [ ]:
summary_md = wf.summary()
print("written:", RUN / "RUN_SUMMARY.md")
print("\n".join(summary_md.splitlines()[:30]))

In [ ]:
# Where everything stands (state: current / stale / failed / not_run)
wf.status()